# 16. Scaled Global Sensemaking: GraphRAG Community Reports vs Vector RAG on the BenchmarkQED Podcast Corpus

**Presentation map:** Slide 33 (graph construction), Slide 34 (community detection → community reports, the
GraphRAG indexing price), Slide 36 (global "sensemaking" questions are where vector RAG is weakest), Slide 38 (decision
rule by question type).

Notebook 6 showed the effect on **6 global questions** over 183 helpdesk tickets — too few to trust. This notebook uses
the data of the original GraphRAG evaluation: the **"Behind the Tech" podcast transcripts** (70 episodes, ≈4.1 M
characters) released by Microsoft in **BenchmarkQED** (`microsoft/benchmark-qed`; data under the Microsoft Research License — non-commercial research use), with the benchmark's own
questions:

- **50 global questions** (e.g. *"How do guests describe transitioning from another field into AI?"*) — answering
  requires themes spread over many episodes. Each comes with **≈19 gold assertions** generated and validated by
  BenchmarkQED's AutoQ (e.g. *"The response should mention a guest who moved from physics to ML"*).
- **50 local questions** (a fact from one episode) as the control: here the graph should not help.

**Systems** (same model and answer budget):
- **(a) Vector RAG** — top-20 chunks (≈12k tokens) by `text-embedding-3-small`.
- **(b) GraphRAG global search** — per chunk the LLM extracts entities and relations with descriptions; Louvain
  communities are detected on the entity graph and split recursively until each has at most 25 entities; the LLM
  writes one **report per community**; at question time every
  group of reports is asked for a partial answer with a helpfulness score (**map**), and the best partial answers are
  combined into the final answer (**reduce**).

**Scoring (two independent methods):**
1. **Assertion pass rate** (BenchmarkQED AutoE): for each answer the LLM judge checks every gold assertion
   (pass / fail). Objective and per question.
2. **Pairwise win rate** (the GraphRAG paper's protocol) on the global questions: the judge compares the two answers on
   *comprehensiveness, diversity, empowerment* and *relevance*, **twice with the order swapped**; a win counts 1, a tie
   0.5.

**Model:** `gpt-6-luna`, no reasoning, via the OpenAI Batch API (`smash_batch.py`); map and reduce are separate Batch
rounds. Cached in `data/benchmarks/benchmarkqed_podcast/cache/`.

In [1]:
import json, os, re, zipfile, random
from collections import Counter, defaultdict
from typing import Literal
import numpy as np
import networkx as nx
from pydantic import BaseModel, Field
from smash_batch import LLM, mode_from_env, wilson, paired_boot

MODE = mode_from_env()
TEST = MODE == "test"
BASE = "data/benchmarks/benchmarkqed_podcast"
llm = LLM(f"{BASE}/cache" + ("_test" if TEST else ""))

z = zipfile.ZipFile(f"{BASE}/raw_data.zip")
episodes = sorted((json.loads(z.read(n)) for n in z.namelist() if n.endswith(".json")), key=lambda e: e["episode"])
GQ = json.load(open(f"{BASE}/data_global_questions_assertions.json"))
LQ = json.load(open(f"{BASE}/data_local_questions_assertions.json"))
if TEST:
    episodes, GQ, LQ = episodes[:6], GQ[:3], []
    LQ = [q for q in json.load(open(f"{BASE}/data_local_questions_assertions.json"))
          if any(e["episode"].split(" ")[0] in q["question_text"] or e["episode"].split("with ")[-1] in q["question_text"] for e in episodes)][:3]
print(f"{len(episodes)} episodes, {sum(len(e['text']) for e in episodes):,} characters | "
      f"{len(GQ)} global questions ({np.mean([len(q['assertions']) for q in GQ]):.1f} assertions each), "
      f"{len(LQ)} local questions")
print("Example global question:", GQ[0]["question_text"])
print("  first assertions:", *[a["statement"] for a in GQ[0]["assertions"][:3]], sep="\n   - ")

70 episodes, 4,087,410 characters | 50 global questions (18.9 assertions each), 50 local questions
Example global question: How do guests describe transitioning from one field of study into AI or machine learning?
  first assertions:
   - The response should address transitions into AI or machine learning from non-computing academic backgrounds.
   - The response should address transitions into AI or machine learning through self-directed study of foundational material.
   - The response should address transitions into AI or machine learning through workplace-driven learning rather than formal schooling.


## Chunks and the vector index

In [2]:
CHUNK_CHARS, OVERLAP = 2400, 300
def make_chunks(text):
    out, i = [], 0
    while i < len(text):
        j = min(len(text), i + CHUNK_CHARS)
        if j < len(text):
            k = text.rfind(". ", i + CHUNK_CHARS // 2, j)
            j = k + 1 if k > 0 else j
        out.append(text[i:j].strip())
        if j >= len(text):
            break
        i = j - OVERLAP
    return out

chunks = [f"[{e['episode']}]\n{c}" for e in episodes for c in make_chunks(e["text"])]
C = llm.embed("chunks", chunks)
QS = [("global", q) for q in GQ] + [("local", q) for q in LQ]
Q = llm.embed("questions", [q["question_text"] for _, q in QS])
print(f"{len(chunks):,} chunks")

2,092 chunks


## GraphRAG index: extraction → entity graph → communities → community reports (Slides 33–34)

In [3]:
class Entity(BaseModel):
    name: str
    type: str = Field(description="PERSON, ORGANIZATION, TECHNOLOGY, PRODUCT, CONCEPT, EVENT, PLACE or WORK")
    description: str = Field(description="one or two sentences about the entity, grounded in the text")

class Relation(BaseModel):
    source: str
    target: str
    description: str = Field(description="how source and target are related, grounded in the text")

class Extraction(BaseModel):
    entities: list[Entity]
    relations: list[Relation]

EXTRACT_SYS = ("Extract a knowledge graph from a podcast transcript passage (episode title in brackets). List the "
               "important entities (people, organisations, technologies, products, concepts, events, places, works) "
               "with a description grounded in the passage, and the relations between them. Use full canonical names. "
               "At most 20 entities and 20 relations.")
res = llm.run("extract", {f"c{i}": LLM.body([{"role": "system", "content": EXTRACT_SYS}, {"role": "user", "content": c}],
                                            max_tokens=4000, schema=Extraction) for i, c in enumerate(chunks)}, parse=Extraction)
norm = lambda s: re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()
ENT = {}
G = nx.Graph()
for i in range(len(chunks)):
    ex = res[f"c{i}"]["parsed"] or {"entities": [], "relations": []}
    for e in ex["entities"]:
        k = norm(e["name"])
        if not k:
            continue
        info = ENT.setdefault(k, {"name": e["name"], "type": e["type"], "descs": [], "chunks": set()})
        info["descs"].append(e["description"]); info["chunks"].add(i)
        G.add_node(k)
    for r in ex["relations"]:
        a, b = norm(r["source"]), norm(r["target"])
        if a and b and a != b:
            for k, nm in ((a, r["source"]), (b, r["target"])):
                ENT.setdefault(k, {"name": nm, "type": "?", "descs": [], "chunks": set()})["chunks"].add(i)
            if G.has_edge(a, b):
                G[a][b]["descs"].append(r["description"]); G[a][b]["weight"] += 1
            else:
                G.add_edge(a, b, descs=[r["description"]], weight=1)
print(f"Graph: {G.number_of_nodes():,} entities, {G.number_of_edges():,} relations "
      f"({sum(res[f'c{i}']['parsed'] is None for i in range(len(chunks)))} unparsable extractions)")

MAX_SIZE = 25          # GraphRAG caps community size too (hierarchical Leiden, max_cluster_size = 10)
# Clustering runs on integer node ids in sorted-name order: Louvain iterates over sets, and sets of strings change
# order between Python processes (hash randomisation), which would make the communities irreproducible.
NAMES = sorted(G.nodes())
GI = nx.Graph()
GI.add_nodes_from(range(len(NAMES)))
pos = {n: i for i, n in enumerate(NAMES)}
GI.add_weighted_edges_from(sorted((pos[a], pos[b], d["weight"]) for a, b, d in G.edges(data=True)))
def split(c):
    # Recursively re-cluster communities that are too large to summarise in one report.
    if len(c) <= MAX_SIZE:
        return [c]
    parts = nx.community.louvain_communities(GI.subgraph(sorted(c)), weight="weight", resolution=1.0, seed=7)
    if len(parts) == 1:
        return [c]
    return [x for part in sorted(parts, key=min) for x in split(part)]
top = sorted(nx.community.louvain_communities(GI, weight="weight", resolution=1.0, seed=7), key=min)
comms = [{NAMES[i] for i in c} for t in top for c in split(t) if len(c) >= 3]
comms.sort(key=lambda c: (-sum(G.degree(n) for n in c), min(c)))
print(f"Louvain communities, recursively split to <= {MAX_SIZE} entities, with >= 3 entities: {len(comms)} "
      f"(from {len(top)} top-level communities; largest {len(comms[0])})")

def community_text(c, budget=24000):
    ents = sorted(c, key=lambda n: (-G.degree(n), n))
    lines = ["ENTITIES"] + [f"- {ENT[n]['name']} ({ENT[n]['type']}): {' '.join(dict.fromkeys(ENT[n]['descs']))[:400]}" for n in ents]
    lines += ["RELATIONS"] + [f"- {ENT[a]['name']} — {ENT[b]['name']}: {' '.join(dict.fromkeys(G[a][b]['descs']))[:300]}"
                              for a, b in sorted((tuple(sorted(e)) for e in G.subgraph(c).edges()), key=lambda e: (-G[e[0]][e[1]]["weight"], e))]
    out, n = [], 0
    for l in lines:
        if n + len(l) > budget:
            break
        out.append(l); n += len(l)
    return "\n".join(out)

class Report(BaseModel):
    title: str
    summary: str = Field(description="an executive summary of the community: who and what it is about and why it matters")
    findings: list[str] = Field(description="5-10 key insights, each a few sentences grounded in the entities and relations")

REPORT_SYS = ("You write a community report for a knowledge graph built from podcast transcripts. Given the entities and "
              "relations of one community, write a title, an executive summary and 5-10 key findings. Use only the given "
              "information.")
rep = llm.run("reports", {f"r{i}": LLM.body([{"role": "system", "content": REPORT_SYS},
                                             {"role": "user", "content": community_text(c)}], max_tokens=2500, schema=Report)
                          for i, c in enumerate(comms)}, parse=Report)
REPORTS = []
for i in range(len(comms)):
    r = rep[f"r{i}"]["parsed"]
    if r:
        REPORTS.append(f"# {r['title']}\n{r['summary']}\n" + "\n".join(f"- {f}" for f in r["findings"]))
print(f"{len(REPORTS)} community reports, {np.mean([len(r) for r in REPORTS]) / 4:.0f} tokens each on average")
print("\nExample report:\n" + REPORTS[0][:900])

Graph: 11,319 entities, 18,737 relations (0 unparsable extractions)


Louvain communities, recursively split to <= 25 entities, with >= 3 entities: 1185 (from 198 top-level communities; largest 273)


1185 community reports, 373 tokens each on average

Example report:
# Kevin Scott on Technology, Creativity, and Public Benefit
This community centers on Kevin Scott, host of Behind the Tech and Microsoft’s Chief Technology Officer, and spans his conversations about computing, AI, careers, creative practice, and society. It connects his technical background in compilers and high-performance computing with interests such as pottery and classical piano, while emphasizing that technology should serve people and contribute to social good. The breadth of topics highlights how technical work, human expression, education, and public choices intersect.
- Scott’s computing background includes compiler development, assembly language, and high-performance computing. He describes himself as a compiler specialist and recalls learning through practice, studying existing implementations, and building software such as compilers for introductory courses.
- Scott sees AI 


## Answering: vector top-20 vs GraphRAG map-reduce over the community reports

**Map:** reports are packed into groups of ≈8k tokens (in a fixed shuffled order, as GraphRAG does); each group gets
the question and returns the relevant points with a 0–100 helpfulness score. **Reduce:** the points with score > 0,
best first, up to ≈12k tokens, are combined into the final answer.

In [4]:
GROUP_CHARS = 32000
order = list(range(len(REPORTS))); random.Random(7).shuffle(order)
groups, cur, n = [], [], 0
for i in order:
    if cur and n + len(REPORTS[i]) > GROUP_CHARS:
        groups.append(cur); cur, n = [], 0
    cur.append(i); n += len(REPORTS[i])
if cur:
    groups.append(cur)
print(f"{len(groups)} report groups -> {len(groups)} map calls per question")

class Points(BaseModel):
    points: list[str] = Field(description="the points from these reports that help answer the question; empty if none")
    score: int = Field(description="0-100: how helpful these points are for answering the question")

MAP_SYS = ("You help answer a question using a set of community reports from a knowledge graph of podcast "
           "transcripts. Extract the points that help answer the question (with the names of the people and episodes "
           "involved) and rate their helpfulness from 0 to 100. Use only the reports.")
mp = llm.run("map", {f"{qi}|{gi}": LLM.body([{"role": "system", "content": MAP_SYS},
                                             {"role": "user", "content": "REPORTS:\n" + "\n\n".join(REPORTS[i] for i in g) +
                                              f"\n\nQuestion: {q['question_text']}"}], max_tokens=1500, schema=Points)
                     for qi, (_, q) in enumerate(QS) for gi, g in enumerate(groups)}, parse=Points)

ANSWER_SYS = ("Answer the question comprehensively, using only the provided material. Cover the different perspectives "
              "and examples it contains and name the people involved. Do not mention the material itself.")
def reduce_ctx(qi, budget=48000):
    pts = []
    for gi in range(len(groups)):
        p = mp[f"{qi}|{gi}"]["parsed"]
        if p and p["score"] > 0:
            pts += [(p["score"], x) for x in p["points"]]
    out, n = [], 0
    for s, x in sorted(pts, key=lambda t: -t[0]):
        if n + len(x) > budget:
            break
        out.append(f"- (helpfulness {s}) {x}"); n += len(x)
    return "\n".join(out) or "(no relevant points)"

def vec_ctx(qi, k=20):
    top = np.argsort(-(C @ Q[qi]))[:k]
    return "\n\n".join(chunks[i] for i in top)

reqs = {}
for qi, (kind, q) in enumerate(QS):
    reqs[f"vector|{qi}"] = LLM.body([{"role": "system", "content": ANSWER_SYS},
                                     {"role": "user", "content": f"Material (transcript excerpts):\n{vec_ctx(qi)}\n\nQuestion: {q['question_text']}"}], max_tokens=2000)
    reqs[f"graph|{qi}"] = LLM.body([{"role": "system", "content": ANSWER_SYS},
                                    {"role": "user", "content": f"Material (points from community reports):\n{reduce_ctx(qi)}\n\nQuestion: {q['question_text']}"}], max_tokens=2000)
ANS = {k: v["text"] for k, v in llm.run("answers", reqs).items()}
print(f"{len(ANS)} answers; average length vector {np.mean([len(ANS[f'vector|{i}']) for i in range(len(QS))]):,.0f} chars, "
      f"graph {np.mean([len(ANS[f'graph|{i}']) for i in range(len(QS))]):,.0f} chars")

57 report groups -> 57 map calls per question


200 answers; average length vector 2,170 chars, graph 2,970 chars


## Scoring 1: assertion pass rate (BenchmarkQED AutoE)

In [5]:
class Checks(BaseModel):
    passed: list[int] = Field(description="1 if the response satisfies the assertion, else 0; one per assertion, in order")

def check_prompt(q, answer):
    lst = "\n".join(f"{i + 1}. {a['statement']}" for i, a in enumerate(q["assertions"]))
    return ("Check whether a response satisfies each assertion. An assertion is satisfied if the response clearly "
            f"contains the required content.\n\nQuestion: {q['question_text']}\n\nResponse:\n{answer}\n\nAssertions:\n{lst}")
ck = llm.run("assertions", {f"{m}|{qi}": LLM.body([{"role": "user", "content": check_prompt(q, ANS[f"{m}|{qi}"])}],
                                                  max_tokens=800, schema=Checks)
                            for qi, (_, q) in enumerate(QS) for m in ("vector", "graph")}, parse=Checks)
def pass_rate(m, qi):
    n = len(QS[qi][1]["assertions"]); p = (ck[f"{m}|{qi}"]["parsed"] or {"passed": []})["passed"]
    return sum(1 for x in (p + [0] * n)[:n] if x) / n

RES = {}
print(f"{'questions':<10}{'n':>4}{'vector':>10}{'graph':>9}{'graph - vector [95% CI]':>28}")
for kind in ("global", "local"):
    idx = [i for i, (k, _) in enumerate(QS) if k == kind]
    if not idx:
        continue
    v = [pass_rate("vector", i) for i in idx]; g = [pass_rate("graph", i) for i in idx]
    dm, lo, hi = paired_boot(v, g)
    RES[kind] = (len(idx), np.mean(v), np.mean(g), dm, lo, hi)
    print(f"{kind:<10}{len(idx):>4}{np.mean(v):>10.1%}{np.mean(g):>9.1%}{f'{dm:+.1%} [{lo:+.1%}, {hi:+.1%}]':>28}")

questions    n    vector    graph     graph - vector [95% CI]
global      50     56.7%    76.5%     +19.8% [+14.9%, +24.6%]
local       50    100.0%    69.7%     -30.3% [-43.0%, -18.7%]


## Scoring 2: pairwise win rate on the global questions (both orders)

In [6]:
CRITERIA = {"comprehensiveness": "How much detail does the answer provide to cover all aspects and details of the question?",
            "diversity": "How varied and rich is the answer in providing different perspectives and insights on the question?",
            "empowerment": "How well does the answer help the reader understand and make informed judgements about the topic?",
            "relevance": "How well does the answer address the question, without irrelevant content?"}
class Verdicts(BaseModel):
    comprehensiveness: Literal["1", "2", "tie"]
    diversity: Literal["1", "2", "tie"]
    empowerment: Literal["1", "2", "tie"]
    relevance: Literal["1", "2", "tie"]

def pair_prompt(q, a1, a2):
    crit = "\n".join(f"- {k}: {v}" for k, v in CRITERIA.items())
    return (f"Compare two answers to the question on each criterion and say which is better (1, 2 or tie).\n{crit}\n\n"
            f"Question: {q}\n\nANSWER 1:\n{a1}\n\nANSWER 2:\n{a2}")
gidx = [i for i, (k, _) in enumerate(QS) if k == "global"]
pw = llm.run("pairwise", {f"{order}|{qi}": LLM.body([{"role": "user", "content": pair_prompt(
    QS[qi][1]["question_text"], *( (ANS[f"graph|{qi}"], ANS[f"vector|{qi}"]) if order == "gv" else (ANS[f"vector|{qi}"], ANS[f"graph|{qi}"]) ))}],
    max_tokens=100, schema=Verdicts) for qi in gidx for order in ("gv", "vg")}, parse=Verdicts)

WIN = {}
for crit in CRITERIA:
    scores = []
    for qi in gidx:
        s = 0.0
        for order in ("gv", "vg"):
            v = (pw[f"{order}|{qi}"]["parsed"] or {}).get(crit, "tie")
            graph_pos = "1" if order == "gv" else "2"
            s += 1.0 if v == graph_pos else 0.5 if v == "tie" else 0.0
        scores.append(s / 2)
    m = np.mean(scores)
    rs = np.random.default_rng(7); boot = rs.choice(scores, (5000, len(scores))).mean(1)
    WIN[crit] = (m, np.percentile(boot, 2.5), np.percentile(boot, 97.5))
    print(f"{crit:<18} graph win rate {m:5.1%} [{WIN[crit][1]:.0%}-{WIN[crit][2]:.0%}]  (50% = parity)")

comprehensiveness  graph win rate 91.0% [84%-98%]  (50% = parity)
diversity          graph win rate 92.5% [86%-98%]  (50% = parity)
empowerment        graph win rate 90.0% [82%-97%]  (50% = parity)
relevance          graph win rate 45.0% [39%-52%]  (50% = parity)


### One global question, two answers (first 700 characters)

In [7]:
qi = gidx[0]
print("Q:", QS[qi][1]["question_text"])
for m in ("vector", "graph"):
    print(f"\n{m.upper()} (assertions passed {pass_rate(m, qi):.0%}):\n{ANS[f'{m}|{qi}'][:700]}")

Q: How do guests describe transitioning from one field of study into AI or machine learning?

VECTOR (assertions passed 50%):
Guests describe entering AI and machine learning through several routes:

- **A gradual shift from another discipline.** Percy Liang began with algorithmic questions and computer science theory, then moved into machine learning and AI near the end of his undergraduate studies at MIT, around 2004. He was drawn by the chance to connect theory with questions about intelligence, and later studied machine learning and natural-language processing. Eric Horvitz started in neurobiology and medicine at Stanford, but classes in computer science, AI, philosophy of mind, and cognitive psychology led him to conclude that computer science offered a faster path to understanding minds. He left the pursuit o

GRAPH (assertions passed 50%):
Guests describe several routes into AI and machine learning, but the accounts vary in how directly they explain a transition. Some began in n

## The price of global search (Slide 34)

In [8]:
LEDGER = llm.print_ledger()
idx_usd = sum(LEDGER.get(s, {}).get("usd", 0) for s in ("extract", "reports"))
mp_q = LEDGER.get("map", {}); nq = len(QS)
print(f"\nGraphRAG index (extraction + reports): ${idx_usd:.2f}; per question the map step makes {len(groups)} calls "
      f"({(mp_q.get('in', 0) + mp_q.get('out', 0)) / max(nq, 1):,.0f} tokens) before the answer; vector RAG makes one call.")

stage                           calls     input tok   output tok      USD  via Batch
answers                           200     1,548,170       95,195     0.10       100%
assertions                        200       152,808        5,856     0.01       100%
extract                         2,092     1,728,432    1,780,227     0.53       100%
map                             5,800    32,998,606      689,008     1.82       100%
pairwise                          100       207,696        3,200     0.01       100%
reports                         1,197       792,040      332,575     0.12       100%
(embeddings)                        0     1,139,780            0     0.02         0%
TOTAL                           9,589    38,567,532    2,906,061     2.62

GraphRAG index (extraction + reports): $0.65; per question the map step makes 57 calls (336,876 tokens) before the answer; vector RAG makes one call.


In [9]:
print("=" * 92)
print(f"RESULTS SUMMARY — Notebook 16: global sensemaking on the BenchmarkQED podcast corpus, gpt-6-luna, {MODE} run")
print("=" * 92)
for kind, (n, v, g, dm, lo, hi) in RES.items():
    verdict = "graph better" if lo > 0 else "vector better" if hi < 0 else "no resolvable difference"
    print(f"{kind:<7} questions (n={n}): assertions passed vector {v:.1%} | graph {g:.1%} | "
          f"diff {dm:+.1%} [{lo:+.1%}, {hi:+.1%}] -> {verdict}")
print("Pairwise (global questions, graph vs vector, both orders): " +
      "; ".join(f"{c} {m:.0%} [{lo:.0%}-{hi:.0%}]" for c, (m, lo, hi) in WIN.items()))
print(f"Index: {len(chunks):,} chunks -> {G.number_of_nodes():,} entities -> {len(REPORTS)} community reports; "
      f"{len(groups)} map calls per question")
print("\nLecture message (Slides 33-38): global questions are the case GraphRAG was designed for. The assertion row"
      "\nsays whether its answers actually contain more of the expected content, the pairwise row whether a judge prefers"
      "\nthem, and the local row is the control where the graph should not be needed.")

RESULTS SUMMARY — Notebook 16: global sensemaking on the BenchmarkQED podcast corpus, gpt-6-luna, batch run
global  questions (n=50): assertions passed vector 56.7% | graph 76.5% | diff +19.8% [+14.9%, +24.6%] -> graph better
local   questions (n=50): assertions passed vector 100.0% | graph 69.7% | diff -30.3% [-43.0%, -18.7%] -> vector better
Pairwise (global questions, graph vs vector, both orders): comprehensiveness 91% [84%-98%]; diversity 92% [86%-98%]; empowerment 90% [82%-97%]; relevance 45% [39%-52%]
Index: 2,092 chunks -> 11,319 entities -> 1185 community reports; 57 map calls per question

Lecture message (Slides 33-38): global questions are the case GraphRAG was designed for. The assertion row
says whether its answers actually contain more of the expected content, the pairwise row whether a judge prefers
them, and the local row is the control where the graph should not be needed.


## Added after review: GraphRAG local search on local questions

Global search (above) answers every question from the community reports alone, which is the wrong mode for a question
about one episode. GraphRAG has a second mode for such questions, **local search**, which this section adds on the same
index (no new extraction):

1. the question is embedded and matched to the **10 entities** whose name + description are most similar;
2. a context of the same size as the vector baseline's (≈48,000 characters ≈ 12k tokens) is filled in GraphRAG's default
   proportions: **15%** community reports of the leaf communities that contain the matched entities (most matched
   entities first), **35%** entity descriptions and their relations (relations between matched entities first, then the
   strongest other relations), **50%** source chunks linked to the matched entities (chunks linked to the most matched
   entities first);
3. the LLM answers once with the same answer prompt; the same assertion judge scores the answer.

Local search is run on both the local and the global questions.

In [10]:
LOCAL_CHARS = 48000
PROP = {"reports": 0.15, "entities": 0.35, "sources": 0.50}      # GraphRAG local-search context proportions
EKEYS = sorted(ENT)
ETXT = [f"{ENT[k]['name']}: {' '.join(dict.fromkeys(ENT[k]['descs']))[:600]}" for k in EKEYS]
EV = llm.embed("entities", ETXT)
COMM_OF = defaultdict(list)                                      # entity -> leaf communities with a report
for i, c in enumerate(comms):
    if rep[f"r{i}"]["parsed"]:
        for n in c:
            COMM_OF[n].append(i)
REPORT_OF = {i: f"# {r['title']}\n{r['summary']}\n" + "\n".join(f"- {f}" for f in r["findings"])
             for i in range(len(comms)) if (r := rep[f"r{i}"]["parsed"])}

def fill(lines, budget):
    out, n = [], 0
    for l in lines:
        if n + len(l) > budget:
            continue
        out.append(l); n += len(l)
    return out

def local_ctx(qi, k=10):
    top = [EKEYS[i] for i in np.argsort(-(EV @ Q[qi]))[:k]]
    rank = {e: r for r, e in enumerate(top)}
    cc = Counter(c for e in top for c in COMM_OF[e])
    reps = fill([REPORT_OF[c] for c, _ in sorted(cc.items(), key=lambda t: (-t[1], t[0]))], PROP["reports"] * LOCAL_CHARS)
    ents = [f"- {ENT[e]['name']} ({ENT[e]['type']}): {' '.join(dict.fromkeys(ENT[e]['descs']))[:600]}" for e in top]
    inside = sorted({tuple(sorted((a, b))) for a in top for b in G[a] if b in rank} if top else [],
                    key=lambda e: (-G[e[0]][e[1]]["weight"], e))
    outside = sorted({tuple(sorted((a, b))) for a in top for b in G[a] if b not in rank},
                     key=lambda e: (min(rank.get(e[0], k), rank.get(e[1], k)), -G[e[0]][e[1]]["weight"], e))
    rels = [f"- {ENT[a]['name']} — {ENT[b]['name']}: {' '.join(dict.fromkeys(G[a][b]['descs']))[:300]}" for a, b in inside + outside]
    er = fill(ents + rels, PROP["entities"] * LOCAL_CHARS)
    hits = Counter(); best = {}
    for e in top:
        for c in ENT[e]["chunks"]:
            hits[c] += 1; best[c] = min(best.get(c, k), rank[e])
    src = fill([chunks[c] for c in sorted(hits, key=lambda c: (-hits[c], best[c], c))], PROP["sources"] * LOCAL_CHARS)
    return ("-----Reports-----\n" + "\n\n".join(reps) + "\n\n-----Entities and relationships-----\n" + "\n".join(er) +
            "\n\n-----Sources-----\n" + "\n\n".join(src))

LCTX = {qi: local_ctx(qi) for qi in range(len(QS))}
LOC = llm.run("local_answers", {f"local|{qi}": LLM.body([{"role": "system", "content": ANSWER_SYS},
    {"role": "user", "content": f"Material (knowledge-graph context and transcript excerpts):\n{LCTX[qi]}\n\nQuestion: {q['question_text']}"}],
    max_tokens=2000) for qi, (_, q) in enumerate(QS)})
ANS.update({k: v["text"] for k, v in LOC.items()})
ck.update(llm.run("local_assertions", {f"local|{qi}": LLM.body([{"role": "user", "content": check_prompt(q, ANS[f"local|{qi}"])}],
                                                              max_tokens=800, schema=Checks)
                                       for qi, (_, q) in enumerate(QS)}, parse=Checks))

ARows = llm.rows("answers")
def mean_in(m, idx, rows=None):
    rows = rows or ARows
    return np.mean([next(r["r"]["in"] for r in rows.values() if r["key"] == f"{m}|{i}") for i in idx])
LRows = llm.rows("local_answers")
print(f"{'questions':<10}{'n':>4}{'vector':>9}{'global':>9}{'local':>9}{'local - vector [95% CI]':>27}{'local - global [95% CI]':>27}")
LRES = {}
for kind in ("local", "global"):
    idx = [i for i, (kk, _) in enumerate(QS) if kk == kind]
    v, g, l = ([pass_rate(m, i) for i in idx] for m in ("vector", "graph", "local"))
    dv, dg = paired_boot(v, l), paired_boot(g, l)
    LRES[kind] = (len(idx), np.mean(v), np.mean(g), np.mean(l), dv, dg)
    print(f"{kind:<10}{len(idx):>4}{np.mean(v):>9.1%}{np.mean(g):>9.1%}{np.mean(l):>9.1%}"
          f"{f'{dv[0]:+.1%} [{dv[1]:+.1%}, {dv[2]:+.1%}]':>27}{f'{dg[0]:+.1%} [{dg[1]:+.1%}, {dg[2]:+.1%}]':>27}")
allq = range(len(QS))
print(f"\nMean prompt tokens per answer call: vector {mean_in('vector', allq):,.0f} | global-search reduce "
      f"{mean_in('graph', allq):,.0f} (+ {len(groups)} map calls) | local search {mean_in('local', allq, LRows):,.0f}")
_ = llm.print_ledger(["local_answers", "local_assertions"])

[local_answers] 100 new requests -> batch (0 cached)


[local_answers] uploaded input file 1/1 (100 requests, 4 MB)


[local_answers] started batch batch_6ad7e615fefc8190a276c84e0277b9c3


[local_answers] 1 min: 1 running, 0 queued, 0 results cached


[local_answers] 6 min: 1 running, 0 queued, 0 results cached


...

[local_assertions] 207 min: 1 running, 0 queued, 0 results cached


[local_assertions] batch batch_6ad7e7a35c588190940045775414faaf completed: 100 ok, 0 failed


[local_assertions] 228 min: 0 running, 0 queued, 100 results cached


[local_assertions] done in 227.9 min


questions    n   vector   global    local    local - vector [95% CI]    local - global [95% CI]
local       50   100.0%    69.7%    95.7%      -4.3% [-10.3%, +0.0%]    +26.0% [+14.0%, +39.0%]
global      50    56.7%    76.5%    41.5%    -15.2% [-19.5%, -10.7%]    -35.0% [-39.9%, -30.1%]

Mean prompt tokens per answer call: vector 10,741 | global-search reduce 4,741 (+ 57 map calls) | local search 8,606
stage                           calls     input tok   output tok      USD  via Batch
local_answers                     100       860,571       27,766     0.05       100%
local_assertions                  100        56,572        2,928     0.00       100%
(embeddings)                        0     1,531,957            0     0.03         0%
TOTAL                             200     2,449,100       30,694     0.08
